In [1]:
import torch
import numpy as np

def cutmix_data(x, y, alpha=1.0):
    '''Apply CutMix augmentation'''
    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size()[0]).to(x.device)
    target_a = y
    target_b = y[rand_index]
    
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[rand_index, :, bbx1:bbx2, bby1:bby2]
    
    # Adjust lambda to match the exact area ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    return x, target_a, target_b, lam

def rand_bbox(size, lam):
    '''Generate random bounding box'''
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import numpy as np

# ============ Config ============
experiment_name = "cutmix"
log_dir = f"runs/{experiment_name}"
save_path = f"runs/{experiment_name}/best.pth"
num_epochs = 50
batch_size = 64
learning_rate = 0.001
cutmix_prob = 0.5
cutmix_alpha = 1.0
# ================================

# 1. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. TensorBoard SummaryWriter
writer = SummaryWriter(log_dir)

# 3. Transform 
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),        # Slight random shift
    transforms.RandomHorizontalFlip(),           # Horizontal flip with 50% chance
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), 
                         (0.5, 0.5, 0.5))
])


# 4. Dataset & DataLoader
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# 5. Model (fix warning: use weights=None)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model.to(device)

# 6. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 7. Evaluation function
def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 8. Train one epoch with CutMix
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        r = np.random.rand()
        if r < cutmix_prob:
            lam = np.random.beta(cutmix_alpha, cutmix_alpha)
            rand_index = torch.randperm(images.size()[0]).to(device)
            target_a = labels
            target_b = labels[rand_index]
            bbx1, bby1, bbx2, bby2 = rand_bbox(images.size(), lam)
            images[:, :, bbx1:bbx2, bby1:bby2] = images[rand_index, :, bbx1:bbx2, bby1:bby2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size(-1) * images.size(-2)))
            outputs = model(images)
            loss = lam * criterion(outputs, target_a) + (1 - lam) * criterion(outputs, target_b)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    train_acc = correct / total
    val_acc = evaluate()

    writer.add_scalar("Loss/train", avg_loss, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)
    print(f"Epoch [{epoch+1}] Train Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    return val_acc

# 9. Random bbox helper (used by CutMix)
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

# 10. Training loop with model checkpointing
best_val_acc = 0.0
for epoch in range(num_epochs):
    val_acc = train_one_epoch(epoch)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"✅ Best model saved with val acc: {best_val_acc:.4f}")

writer.close()

Epoch [1] Train Loss: 1.8306, Train Acc: 0.3578, Val Acc: 0.4922
✅ Best model saved with val acc: 0.4922
Epoch [2] Train Loss: 1.5811, Train Acc: 0.4658, Val Acc: 0.5986
✅ Best model saved with val acc: 0.5986
Epoch [3] Train Loss: 1.4380, Train Acc: 0.5212, Val Acc: 0.6313
✅ Best model saved with val acc: 0.6313
Epoch [4] Train Loss: 1.3574, Train Acc: 0.5518, Val Acc: 0.6501
✅ Best model saved with val acc: 0.6501
Epoch [5] Train Loss: 1.3173, Train Acc: 0.5645, Val Acc: 0.6798
✅ Best model saved with val acc: 0.6798
Epoch [6] Train Loss: 1.2749, Train Acc: 0.5804, Val Acc: 0.6900
✅ Best model saved with val acc: 0.6900
Epoch [7] Train Loss: 1.1986, Train Acc: 0.6086, Val Acc: 0.7015
✅ Best model saved with val acc: 0.7015
Epoch [8] Train Loss: 1.2032, Train Acc: 0.6032, Val Acc: 0.7171
✅ Best model saved with val acc: 0.7171
Epoch [9] Train Loss: 1.1970, Train Acc: 0.6065, Val Acc: 0.7395
✅ Best model saved with val acc: 0.7395
Epoch [10] Train Loss: 1.0918, Train Acc: 0.6462, Val A

In [8]:
import subprocess
import time
import webbrowser

def start_tensorboard(logdir="runs", port=6006):
    """
    Starts TensorBoard as a background process and opens it in your default browser.
    Returns the process handle so you can terminate it later.
    """
    tb_cmd = [
        "tensorboard",
        f"--logdir={logdir}",
        f"--port={port}",
        "--host=localhost"
    ]
    
    # Start the process
    print(f"Starting TensorBoard on port {port}...")
    process = subprocess.Popen(tb_cmd)
    
    # Give it a second to get going
    time.sleep(2)
    
    # Open in browser automatically
    url = f"http://localhost:{port}"
    print(f"Opening {url} in your default browser...")
    webbrowser.open(url)
    
    return process


def stop_tensorboard(process):
    """
    Stops the TensorBoard process started by `start_tensorboard()`.
    """
    print("Stopping TensorBoard...")
    process.terminate()  # or process.kill()

In [9]:
# process = start_tensorboard(logdir="runs", port=6012)

In [10]:
# stop_tensorboard(process)
# Uncomment the line below to stop TensorBoard when you're done